# Digital Image Processing Lab
## Lab Task 04
### Geometric Transformations with Custom Inputs

**Name:** Sughra Mumshad

**Roll Number:** 2024-SE-13

**Semester:** Fifth

**Department:** Software Engineering

**University:** The University of Azad Jammu and Kashmir, Muzaffarabad

**Lab Instructor:** Engr. Fahad Nisar

In [43]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
import math

from google.colab import files
from PIL import Image

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

# Get actual uploaded filename
image_name = list(uploaded.keys())[0]

# Read image directly from uploaded data
import numpy as np

image_bytes = np.frombuffer(uploaded[image_name], np.uint8)
original_image = cv2.imdecode(image_bytes, cv2.IMREAD_COLOR)

# Validate image
if original_image is None:
    raise ValueError("Image could not be decoded!")

# Convert BGR to RGB
original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

# Get dimensions
height, width = original_image.shape[:2]

print("Image loaded successfully!")
print("Image Name:", image_name)
print("Width:", width)
print("Height:", height)
print("Array Shape:", original_image.shape)

# Display image
plt.figure(figsize=(7, 5))
plt.imshow(original_image)
plt.title("Original Image")
plt.axis("off")
plt.show()

In [ ]:
def transform(image, operation, **params):

    if image is None or image.size == 0:
        raise ValueError("Invalid or empty image!")

    h, w = image.shape[:2]

    if operation == "scale":

        sx = float(params["sx"])
        sy = float(params["sy"])

        if not np.isfinite(sx) or not np.isfinite(sy):
            raise ValueError("Scale factors must be finite!")

        if sx <= 0 or sy <= 0:
            raise ValueError("Scale factors must be greater than zero!")

        new_w = int(round(w * sx))
        new_h = int(round(h * sy))

        if new_w < 1 or new_h < 1:
            raise ValueError("Output dimensions cannot be empty!")

        if new_w * new_h > 25000000:
            raise ValueError("Output image is too large!")

        interpolation = params.get("interpolation", "linear")

        methods = {
            "nearest": cv2.INTER_NEAREST,
            "linear": cv2.INTER_LINEAR,
            "cubic": cv2.INTER_CUBIC,
            "lanczos": cv2.INTER_LANCZOS4
        }

        if interpolation not in methods:
            raise ValueError("Unsupported interpolation method!")

        return cv2.resize(
            image, (new_w, new_h),
            interpolation=methods[interpolation]
        )

    elif operation == "crop":

        left = params["left"]
        top = params["top"]
        right = params["right"]
        bottom = params["bottom"]

        coordinates = [left, top, right, bottom]

        if any(not isinstance(v, (int, np.integer)) or isinstance(v, bool)
               for v in coordinates):
            raise ValueError("Crop coordinates must be integers!")

        if not (0 <= left < right <= w and 0 <= top < bottom <= h):
            raise ValueError("Invalid crop coordinates!")

        return image[top:bottom, left:right].copy()

    elif operation == "flip":

        direction = params["direction"]

        methods = {
            "horizontal": 1,
            "vertical": 0,
            "both": -1
        }

        if direction not in methods:
            raise ValueError("Invalid flipping direction!")

        return cv2.flip(image, methods[direction])

    elif operation == "translate":

        tx = float(params["tx"])
        ty = float(params["ty"])

        if not np.isfinite(tx) or not np.isfinite(ty):
            raise ValueError("Translation values must be finite!")

        matrix = np.float32([
            [1, 0, tx],
            [0, 1, ty]
        ])

        return cv2.warpAffine(image, matrix, (w, h))

    elif operation == "rotate":

        angle = float(params["angle"])
        expand = params["expand"]

        if not np.isfinite(angle):
            raise ValueError("Angle must be finite!")

        if not isinstance(expand, (bool, np.bool_)):
            raise ValueError("Expand must be True or False!")

        center = (w / 2, h / 2)

        matrix = cv2.getRotationMatrix2D(center, angle, 1.0)

        if expand:
            cos = abs(matrix[0, 0])
            sin = abs(matrix[0, 1])

            new_w = int(np.ceil(h * sin + w * cos))
            new_h = int(np.ceil(h * cos + w * sin))

            if new_w * new_h > 25000000:
                raise ValueError("Output image is too large!")

            matrix[0, 2] += new_w / 2 - center[0]
            matrix[1, 2] += new_h / 2 - center[1]

        else:
            new_w, new_h = w, h

        return cv2.warpAffine(
            image, matrix, (new_w, new_h)
        )

    else:
        raise ValueError("Unknown transformation operation!")


print("Reusable transform function created successfully!")

In [ ]:
# Proportional enlargement
enlarged = transform(
    original_image,
    "scale",
    sx=1.5,
    sy=1.5,
    interpolation="linear"
)

# Proportional reduction
reduced = transform(
    original_image,
    "scale",
    sx=0.5,
    sy=0.5,
    interpolation="linear"
)

# Non-proportional scaling
non_proportional = transform(
    original_image,
    "scale",
    sx=1.5,
    sy=0.8,
    interpolation="linear"
)

# Display results
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

images = [
    original_image,
    enlarged,
    reduced,
    non_proportional
]

titles = [
    "Original Image",
    "Enlarged (1.5x)",
    "Reduced (0.5x)",
    "Non-Proportional"
]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Original dimensions:", original_image.shape)
print("Enlarged dimensions:", enlarged.shape)
print("Reduced dimensions:", reduced.shape)
print("Non-proportional dimensions:", non_proportional.shape)

In [ ]:
height, width = original_image.shape[:2]

print("Image Width:", width)
print("Image Height:", height)

print("Left:", 0)
print("Top:", 0)
print("Right:", width // 2)
print("Bottom:", height // 2)

In [ ]:
# Custom cropping coordinates
left = 0
top = 0
right = width // 2
bottom = height // 2

# Apply cropping
cropped_image = transform(
    original_image,
    "crop",
    left=left,
    top=top,
    right=right,
    bottom=bottom
)

# Display original and cropped images
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(original_image)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cropped_image)
plt.title("Cropped Image")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Original Shape:", original_image.shape)
print("Cropped Shape:", cropped_image.shape)

In [ ]:
# Horizontal flipping
horizontal_flip = transform(
    original_image,
    "flip",
    direction="horizontal"
)

# Vertical flipping
vertical_flip = transform(
    original_image,
    "flip",
    direction="vertical"
)

# Both directions flipping
both_flip = transform(
    original_image,
    "flip",
    direction="both"
)

# Display all results
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

images = [
    original_image,
    horizontal_flip,
    vertical_flip,
    both_flip
]

titles = [
    "Original Image",
    "Horizontal Flip",
    "Vertical Flip",
    "Both Directions"
]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Move image to the right
right_shift = transform(
    original_image,
    "translate",
    tx=50,
    ty=0
)

# Move image to the left
left_shift = transform(
    original_image,
    "translate",
    tx=-50,
    ty=0
)

# Move image upwards
up_shift = transform(
    original_image,
    "translate",
    tx=0,
    ty=-50
)

# Move image downwards
down_shift = transform(
    original_image,
    "translate",
    tx=0,
    ty=50
)

# Display all results
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

images = [
    original_image,
    right_shift,
    left_shift,
    up_shift,
    down_shift
]

titles = [
    "Original",
    "Right +50",
    "Left -50",
    "Up -50",
    "Down +50"
]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Positive angle rotation
positive_rotation = transform(
    original_image,
    "rotate",
    angle=45,
    expand=False
)

# Negative angle rotation
negative_rotation = transform(
    original_image,
    "rotate",
    angle=-45,
    expand=False
)

# Rotation with canvas expansion
expanded_rotation = transform(
    original_image,
    "rotate",
    angle=45,
    expand=True
)

# Display results
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

images = [
    original_image,
    positive_rotation,
    negative_rotation,
    expanded_rotation
]

titles = [
    "Original Image",
    "Positive Rotation +45",
    "Negative Rotation -45",
    "Canvas Expanded"
]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Original Shape:", original_image.shape)
print("Positive Rotation:", positive_rotation.shape)
print("Negative Rotation:", negative_rotation.shape)
print("Expanded Rotation:", expanded_rotation.shape)

In [ ]:
# Function to test invalid inputs safely

def test_invalid_input(description, operation, **params):

    try:
        transform(original_image, operation, **params)

        print(description, ": FAILED - Invalid input accepted")

    except (ValueError, TypeError, KeyError) as error:

        print(description, ": PASSED -", error)


# Test 1: Invalid scaling
test_invalid_input(
    "Invalid Scaling",
    "scale",
    sx=-1,
    sy=1
)

# Test 2: Invalid cropping
test_invalid_input(
    "Invalid Cropping",
    "crop",
    left=100,
    top=100,
    right=50,
    bottom=50
)

# Test 3: Invalid flipping
test_invalid_input(
    "Invalid Flipping",
    "flip",
    direction="diagonal"
)

# Test 4: Invalid translation
test_invalid_input(
    "Invalid Translation",
    "translate",
    tx=float("nan"),
    ty=10
)

# Test 5: Invalid rotation
test_invalid_input(
    "Invalid Rotation",
    "rotate",
    angle=float("inf"),
    expand=False
)

In [ ]:
# Test 1: Zero Translation
zero_translation = transform(
    original_image,
    "translate",
    tx=0,
    ty=0
)

# Test 2: Scale factors of one
same_scale = transform(
    original_image,
    "scale",
    sx=1,
    sy=1
)

# Test 3: Zero Rotation
zero_rotation = transform(
    original_image,
    "rotate",
    angle=0,
    expand=False
)

# Compare images
tests = [
    ("Zero Translation", original_image, zero_translation),
    ("Scale Factors = 1", original_image, same_scale),
    ("Zero Rotation", original_image, zero_rotation)
]

for name, original, result in tests:
    passed = np.array_equal(original, result)
    print(name, ":", "PASS" if passed else "FAIL")

In [ ]:
# Enlarge the same image using four methods

nearest = transform(
    original_image, "scale",
    sx=2, sy=2,
    interpolation="nearest"
)

bilinear = transform(
    original_image, "scale",
    sx=2, sy=2,
    interpolation="linear"
)

bicubic = transform(
    original_image, "scale",
    sx=2, sy=2,
    interpolation="cubic"
)

lanczos = transform(
    original_image, "scale",
    sx=2, sy=2,
    interpolation="lanczos"
)

# Display comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

results = [
    (nearest, "Nearest Neighbour"),
    (bilinear, "Bilinear"),
    (bicubic, "Bicubic"),
    (lanczos, "Lanczos")
]

for ax, (img, title) in zip(axes.flat, results):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# @title Default title text
def run_custom_transform():

    print("GEOMETRIC TRANSFORMATION MENU")
    print("1. Scaling")
    print("2. Cropping")
    print("3. Flipping")
    print("4. Translation")
    print("5. Rotation")

    try:
        choice = input("Select operation (1-5): ").strip()

        if choice == "1":
            sx = float(input("Enter horizontal scale: "))
            sy = float(input("Enter vertical scale: "))
            method = input(
                "Interpolation (nearest/linear/cubic/lanczos): "
            ).strip().lower()

            result = transform(
                original_image, "scale",
                sx=sx, sy=sy, interpolation=method
            )

        elif choice == "2":
            left = int(input("Enter left: "))
            top = int(input("Enter top: "))
            right = int(input("Enter right: "))
            bottom = int(input("Enter bottom: "))

            result = transform(
                original_image, "crop",
                left=left, top=top,
                right=right, bottom=bottom
            )

        elif choice == "3":
            direction = input(
                "Enter direction (horizontal/vertical/both): "
            ).strip().lower()

            result = transform(
                original_image, "flip",
                direction=direction
            )

        elif choice == "4":
            tx = float(input("Enter horizontal offset: "))
            ty = float(input("Enter vertical offset: "))

            result = transform(
                original_image, "translate",
                tx=tx, ty=ty
            )

        elif choice == "5":
            angle = float(input("Enter rotation angle: "))
            expand_text = input(
                "Expand canvas? (yes/no): "
            ).strip().lower()

            if expand_text not in ["yes", "no"]:
                raise ValueError("Enter yes or no!")

            result = transform(
                original_image, "rotate",
                angle=angle,
                expand=(expand_text == "yes")
            )

        else:
            raise ValueError("Unknown operation selected!")

        print("\nTransformation completed successfully!")
        print("Output dimensions:", result.shape)

        plt.figure(figsize=(7, 5))
        plt.imshow(result)
        plt.title("Custom Transformed Image")
        plt.axis("off")
        plt.show()

        return result

    except (ValueError, TypeError, KeyError, OverflowError) as error:
        print("Input Error:", error)
        print("Please correct your input and try again.")
        return None


custom_result = run_custom_transform()

In [ ]:
def test_invalid_input(description, operation, **params):
    try:
        transform(original_image, operation, **params)
        print(description, ": FAILED - Invalid input accepted")

    except (ValueError, TypeError, KeyError, OverflowError) as error:
        print(description, ": PASSED -", error)


# Testing invalid inputs for all operations

test_invalid_input(
    "Invalid Scaling",
    "scale",
    sx=-1,
    sy=1
)

test_invalid_input(
    "Invalid Cropping",
    "crop",
    left=100,
    top=100,
    right=50,
    bottom=50
)

test_invalid_input(
    "Invalid Flipping",
    "flip",
    direction="diagonal"
)

test_invalid_input(
    "Invalid Translation",
    "translate",
    tx=float("nan"),
    ty=10
)

test_invalid_input(
    "Invalid Rotation",
    "rotate",
    angle=float("inf"),
    expand=False
)

In [ ]:
# Store all transformed images
results = {
    "Original": original_image,
    "Enlarged": enlarged,
    "Reduced": reduced,
    "Non_Proportional": non_proportional,
    "Cropped": cropped_image,
    "Horizontal_Flip": horizontal_flip,
    "Vertical_Flip": vertical_flip,
    "Both_Flip": both_flip,
    "Right_Translation": right_shift,
    "Left_Translation": left_shift,
    "Up_Translation": up_shift,
    "Down_Translation": down_shift,
    "Positive_Rotation": positive_rotation,
    "Negative_Rotation": negative_rotation,
    "Expanded_Rotation": expanded_rotation,
    "Nearest": nearest,
    "Bilinear": bilinear,
    "Bicubic": bicubic,
    "Lanczos": lanczos
}

# Display images with labels and dimensions
fig, axes = plt.subplots(5, 4, figsize=(18, 20))

for ax, (name, img) in zip(axes.flat, results.items()):
    ax.imshow(img)
    ax.set_title(f"{name}\nSize: {img.shape[1]} x {img.shape[0]}")
    ax.axis("off")

for ax in axes.flat[len(results):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import shutil


In [ ]:
import os

os.makedirs("transformed_images", exist_ok=True)

print("Folder created successfully!")

In [ ]:
import cv2

images_to_save = {
    "original.jpg": original_image,
    "enlarged.jpg": enlarged,
    "reduced.jpg": reduced,
    "cropped.jpg": cropped_image,
    "horizontal_flip.jpg": horizontal_flip,
    "vertical_flip.jpg": vertical_flip,
    "both_flip.jpg": both_flip,
    "right_shift.jpg": right_shift,
    "left_shift.jpg": left_shift,
    "up_shift.jpg": up_shift,
    "down_shift.jpg": down_shift,
    "positive_rotation.jpg": positive_rotation,
    "negative_rotation.jpg": negative_rotation,
    "expanded_rotation.jpg": expanded_rotation
}

for filename, image in images_to_save.items():
    cv2.imwrite(
        os.path.join("transformed_images", filename),
        cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    )

print("All images saved successfully!")

In [ ]:
import shutil

shutil.make_archive(
    "2024-SE-13_Lab04_Images",
    "zip",
    "transformed_images"
)

print("ZIP file created successfully!")

In [ ]:
from google.colab import files

files.download("2024-SE-13_Lab04_Images.zip")